# Raw trajectories

Polyline subsets of particle tracks on maps. All regimes overlaid per
panel, one colour per regime. Scopes: per HELCOM release subbasin,
German waters, per release quarter (JFM/AMJ/JAS/OND).

In [1]:
import warnings

import dask
import numpy as np
import xarray as xr
import geopandas as gpd
import shapely
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import cartopy.crs as ccrs
from pathlib import Path

In [2]:
def assign_release_subbasin(ds, subbasins):
    # Lazy per (trajectory,) chunk; STRtree built once, looked up per-chunk.
    # The full-sweep concat reaches 60M+ trajectories — eager would OOM.
    tree = shapely.STRtree(subbasins.geometry.values)
    names = subbasins["subbasin"].to_numpy()

    def _lookup(lon, lat):
        out = np.full(lon.shape, None, dtype=object)
        valid = ~(np.isnan(lon) | np.isnan(lat))
        if valid.any():
            pts = shapely.points(lon[valid], lat[valid])
            out[valid] = names[tree.nearest(pts)]
        return out

    lon0 = ds.lon.isel(obs=0, drop=True)
    lat0 = ds.lat.isel(obs=0, drop=True)
    subbasin = xr.apply_ufunc(
        _lookup, lon0, lat0,
        dask="parallelized", output_dtypes=[object],
    )
    return ds.assign(subbasin=subbasin)

# Parameters

In [3]:
data_root = "../data"
output_root = "../output"

n_traj_subset = 300

lon_min, lon_max = 5, 32
lat_min, lat_max = 53, 66
de_lon_min, de_lon_max = 8, 15
de_lat_min, de_lat_max = 53.2, 55.5

baltic_panel_height_in = 2
de_panel_height_in = 1.5

# Regime colours.
regime_colors = {
    "bottom": "tab:orange",
    "surface": "tab:blue",
    "surface_stokes": "tab:green",
}

# Global RNG

In [4]:
seed = np.random.randint(0, 2**31 - 1)
print(f"RNG seed: {seed}")
rng = np.random.default_rng(seed)

RNG seed: 566796349


# Dask cluster

Connect to an external scheduler when ``SCHEDULER_FILE`` is set (written
by the multi-task SLURM job). Otherwise spin up a local cluster on the
current node.

In [5]:
import os
import time
from dask.distributed import Client

scheduler_file = os.environ.get("SCHEDULER_FILE")
if scheduler_file:
    for _ in range(60):
        if os.path.exists(scheduler_file):
            break
        time.sleep(1)
    client = Client(scheduler_file=scheduler_file)
else:
    client = Client(ip="0.0.0.0")
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://192.168.178.47:8787/status,
Dashboard: http://192.168.178.47:8787/status,Workers: 4
Total threads: 12,Total memory: 36.00 GiB
Status: running,Using processes: True
Comm: tcp://192.168.178.47:60207,Workers: 0
Dashboard: http://192.168.178.47:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://192.168.178.47:60219,Total threads: 3
Dashboard: http://192.168.178.47:60220/status,Memory: 9.00 GiB
Nanny: tcp://192.168.178.47:60210,


# Release area

In [6]:
data_root = Path(data_root)
output_root = Path(output_root)

release_area = gpd.read_file(
    data_root / "fucus_redlist_shapefile" / "REDLIST_SIS_Macrophytes.shp"
)
release_area = release_area.loc[
    release_area.F_vesiculo != 0, ["geometry", "CELLCODE"]
].to_crs(crs=ccrs.Geodetic())
release_area

,geometry,CELLCODE
49699,"POLYGON ((20.14019 60.35963, 20.16755 60.44855...",10kmE488N418
49700,"POLYGON ((20.20947 59.99111, 20.23662 60.08001...",10kmE489N414
49701,"POLYGON ((20.31906 60.34667, 20.34688 60.43555...",10kmE489N418
49702,"POLYGON ((20.7398 59.95138, 20.7683 60.04015, ...",10kmE492N414
49703,"POLYGON ((20.85486 60.30645, 20.88407 60.39521...",10kmE492N418
...,...,...
51213,"POLYGON ((23.80809 58.12399, 23.8422 58.21194,...",10kmE513N397
51214,"POLYGON ((24.21081 58.26466, 24.24609 58.35249...",10kmE515N399
51215,"POLYGON ((24.99658 60.10714, 25.0365 60.19477,...",10kmE515N420
51216,"POLYGON ((24.30702 58.07114, 24.34228 58.15895...",10kmE516N397


# HELCOM subbasins

In [7]:
subbasins = gpd.read_file(
    data_root / "helcom_subbasins" / "HELCOM_subbasins_2022_level2.shp"
).to_crs(crs=ccrs.Geodetic()).rename(dict(level_2="subbasin"), axis=1)
subbasins

/Users/wrath/src/github.com/geomar-od-lagrange/2025_fucus-dispersal/.pixi/envs/default/lib/python3.14/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D Polygon' is converted to 'Polygon Z'
  return ogr_read(


,subbasin,HELCOM_ID,Shape_Leng,area_new_k,Shape_Le_1,Shape_Area,geometry
0,Åland Sea,SEA-014,1.686425e+07,16828.264722,3.386842e+07,6.749575e+10,"MULTIPOLYGON Z (((20.9365 59.91963 0, 20.93633..."
1,Arkona Basin,SEA-006,2.290855e+06,17764.929070,3.959215e+06,5.364922e+10,"MULTIPOLYGON Z (((13.2923 54.26615 0, 13.29226..."
2,Bay of Mecklenburg,SEA-005,7.324726e+05,4645.652906,1.250833e+06,1.361923e+10,"POLYGON Z ((11.66527 54.66746 0, 11.66528 54.6..."
3,Bornholm Basin,SEA-007,2.732431e+06,41900.014778,4.771873e+06,1.282214e+11,"POLYGON Z ((17.68075 56.19858 0, 17.66269 56.1..."
4,Bothnian Bay,SEA-017,6.026798e+06,32199.911900,1.422592e+07,1.770893e+11,"POLYGON Z ((24.54719 65.79175 0, 24.54699 65.7..."
5,Bothnian Sea,SEA-015,7.719175e+06,59348.461406,1.627043e+07,2.644347e+11,"MULTIPOLYGON Z (((18.52722 60.27644 0, 18.5259..."
6,Eastern Gotland Basin,SEA-009,2.034451e+06,75133.040876,3.706074e+06,2.467002e+11,"MULTIPOLYGON Z (((21.12929 55.65592 0, 21.1456..."
7,Gdansk Basin,SEA-008,6.849642e+05,5875.877015,1.181195e+06,1.750562e+10,"POLYGON Z ((19.9833 54.96354 0, 19.97228 54.95..."
8,Gulf of Finland,SEA-013,7.609506e+06,29876.095204,1.523829e+07,1.185386e+11,"MULTIPOLYGON Z (((24.98083 59.64038 0, 24.9808..."
9,Gulf of Riga,SEA-011,2.567918e+06,18781.905550,4.888981e+06,6.649281e+10,"MULTIPOLYGON Z (((22.40899 58.21927 0, 22.4089..."


# Load all regimes and attach metadata

In [ ]:
trajectory_root = output_root / "Trajectories"
regimes = sorted(p.name for p in trajectory_root.iterdir() if p.is_dir())
print(f"Regimes: {regimes}")

regime_dsets = {}
for regime in regimes:
    zarr_files = sorted((trajectory_root / regime).glob("**/*.zarr"))
    print(f"{regime}: {len(zarr_files)} trajectory files")
    ds = xr.concat([xr.open_zarr(z) for z in zarr_files], dim="trajectory")
    # First-step displacement of zero ⇒ trajectory was seeded on land.
    ds = ds.where(~(
        (ds.lon.diff("obs").isel(obs=0, drop=True) == 0)
        & (ds.lat.diff("obs").isel(obs=0, drop=True) == 0)
    ))
    ds = ds.assign(release_quarter=ds.time.isel(obs=0, drop=True).dt.quarter)
    ds = assign_release_subbasin(ds, subbasins)
    regime_dsets[regime] = ds

# Precompute per-trajectory scope keys per regime

`release_quarter` / `subbasin` are lazy 1-D `(trajectory,)` arrays.
Compute them once per regime so the per-panel loops below don't re-walk
the graph.

In [ ]:
regime_keys = {}
for regime, ds in regime_dsets.items():
    quarter_np, subbasin_np = dask.compute(ds.release_quarter, ds.subbasin)
    regime_keys[regime] = dict(
        quarter=quarter_np.values,
        subbasin=subbasin_np.values,
    )

# Plot helpers

In [ ]:
def lonlat_aspect(extent):
    """Displayed width / height ratio for a lon/lat ``extent`` with an
    aspect that keeps 1 deg lon at ``lat_mean`` visually equal to 1 deg
    lat. Matches ``ax.set_aspect(1 / cos(lat_mean))`` below."""
    lon_min_, lon_max_, lat_min_, lat_max_ = extent
    lat_mean = 0.5 * (lat_min_ + lat_max_)
    return ((lon_max_ - lon_min_) * np.cos(np.radians(lat_mean))) / (
        lat_max_ - lat_min_
    )


baltic_extent = [lon_min, lon_max, lat_min, lat_max]
de_extent = [de_lon_min, de_lon_max, de_lat_min, de_lat_max]
baltic_aspect = lonlat_aspect(baltic_extent)
de_aspect = lonlat_aspect(de_extent)


def plot_lines(ds_plot, ax, color, lw=None):
    if ds_plot.sizes["trajectory"] == 0:
        return
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        for i in range(ds_plot.sizes["trajectory"]):
            ax.plot(
                ds_plot.lon.isel(trajectory=i),
                ds_plot.lat.isel(trajectory=i),
                color=color, linewidth=lw, alpha=0.3,
                transform=ccrs.PlateCarree(),
            )

# Per HELCOM release subbasin

Subbasin list comes from the union over regimes (they should agree, but
the union is robust to a regime missing a subbasin entirely).

In [ ]:
subbasins_list = sorted({
    s
    for regime in regimes
    for s in regime_keys[regime]["subbasin"]
    if isinstance(s, str)
})

ncols = 4
nrows = int(np.ceil(len(subbasins_list) / ncols))
for regime in regimes:
    ds = regime_dsets[regime]
    fig, axes = plt.subplots(
        nrows=nrows, ncols=ncols,
        figsize=(baltic_panel_height_in * baltic_aspect * ncols, baltic_panel_height_in * nrows),
        layout="constrained",
        subplot_kw=dict(projection=ccrs.PlateCarree()),
    )
    for ax, basin in zip(axes.flat, subbasins_list):
        mask = regime_keys[regime]["subbasin"] == basin
        avail = np.flatnonzero(mask)
        if avail.size == 0:
            ax.set_visible(False)
            continue
        idx = rng.choice(avail, size=min(n_traj_subset, avail.size), replace=False)
        ds_plot = ds.isel(trajectory=idx).compute()
        plot_lines(ds_plot, ax, color=regime_colors[regime], lw=0.5)
        ax.set_extent(baltic_extent, crs=ccrs.PlateCarree())
        ax.coastlines()
        ax.set_title(basin)
    for ax in axes.flat[len(subbasins_list):]:
        ax.set_visible(False)
    fig.suptitle(regime)
    fig.legend(
        handles=[Line2D([], [], color=regime_colors[r], label=r, linewidth=1.5) for r in regimes],
        loc="lower center", ncol=len(regimes),
    )
    plt.show()

# German waters

In [ ]:
fig, axes = plt.subplots(
    nrows=1, ncols=len(regimes),
    figsize=(de_panel_height_in * de_aspect * len(regimes), de_panel_height_in),
    layout="constrained",
    subplot_kw=dict(projection=ccrs.PlateCarree()),
)
for ax, regime in zip(axes, regimes):
    ds = regime_dsets[regime]
    idx = rng.choice(ds.sizes["trajectory"], size=min(n_traj_subset, ds.sizes["trajectory"]), replace=False)
    ds_plot = ds.isel(trajectory=idx).compute()
    plot_lines(ds_plot, ax, color=regime_colors[regime])
    ax.set_extent(de_extent, crs=ccrs.PlateCarree())
    ax.coastlines()
    ax.set_title(regime)
fig.legend(
    handles=[Line2D([], [], color=regime_colors[r], label=r, linewidth=1.5) for r in regimes],
    loc="lower center", ncol=len(regimes),
)
plt.show()

# Per release quarter (JFM/AMJ/JAS/OND)

In [ ]:
quarter_labels = {1: "JFM", 2: "AMJ", 3: "JAS", 4: "OND"}
nrows = len(quarter_labels)
ncols = len(regimes)
fig, axes = plt.subplots(
    nrows=nrows, ncols=ncols,
    figsize=(baltic_panel_height_in * baltic_aspect * ncols, baltic_panel_height_in * nrows),
    layout="constrained",
    subplot_kw=dict(projection=ccrs.PlateCarree()),
)
for row, (q_int, q_label) in enumerate(quarter_labels.items()):
    for col, regime in enumerate(regimes):
        ax = axes[row, col]
        ds = regime_dsets[regime]
        mask = regime_keys[regime]["quarter"] == q_int
        avail = np.flatnonzero(mask)
        if avail.size == 0:
            ax.set_visible(False)
            continue
        idx = rng.choice(avail, size=min(n_traj_subset, avail.size), replace=False)
        ds_plot = ds.isel(trajectory=idx).compute()
        plot_lines(ds_plot, ax, color=regime_colors[regime])
        ax.set_extent(baltic_extent, crs=ccrs.PlateCarree())
        ax.coastlines()
        if row == 0:
            ax.set_title(regime)
        if col == 0:
            ax.set_ylabel(q_label)
fig.legend(
    handles=[Line2D([], [], color=regime_colors[r], label=r, linewidth=1.5) for r in regimes],
    loc="lower center", ncol=len(regimes),
)
plt.show()